# ⚙️ Day 01a: Processes — The Heart of an Operating System

---

## 🎯 What You'll Master Today
- What is a process? Program vs process vs thread
- Process states & lifecycle transitions
- PCB (Process Control Block) structure
- Context switching — how the OS juggles processes
- Process creation: fork/exec, zombie & orphan processes

---

## 🎭 The Analogy

**Program** = a recipe (instructions on paper, passive)  
**Process** = actually cooking the recipe (active, has state: ingredients on counter, oven temperature, current step)  
**Thread** = each chef's hand doing a task in the same kitchen (shared space, same recipe)

> A program becomes a process when it's loaded into memory and starts executing.

---

```
╔══════════════════════════════════════════════════════════════╗
║                   PROCESS STATE DIAGRAM                      ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║   NEW ──admitted──→ READY ──dispatch──→ RUNNING              ║
║                      ↑                    │  │               ║
║                      │                    │  │               ║
║            I/O done  │    I/O or event    │  exit             ║
║            or event  │      wait          │  │               ║
║                      │                    ↓  ↓               ║
║                   WAITING              TERMINATED            ║
║                                                              ║
║   Also: RUNNING ──preempt──→ READY (time quantum expires)   ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Process States — Simulation
# ============================================

class Process:
    STATES = ["NEW", "READY", "RUNNING", "WAITING", "TERMINATED"]
    
    def __init__(self, pid: int, name: str):
        self.pid = pid
        self.name = name
        self.state = "NEW"
        self.program_counter = 0
        self.registers = {}
        self.memory_limits = (0, 0)
        self.open_files = []
    
    def transition(self, new_state: str, reason: str = ""):
        old = self.state
        self.state = new_state
        print(f"  PID {self.pid} ({self.name}): {old} → {new_state}  [{reason}]")
    
    def __str__(self):
        return f"Process(pid={self.pid}, name={self.name}, state={self.state})"


# Lifecycle walkthrough
p = Process(101, "vim")
print(f"Created: {p}\n")

p.transition("READY", "loaded into memory")
p.transition("RUNNING", "CPU scheduler dispatched")
p.transition("WAITING", "waiting for disk I/O")
p.transition("READY", "I/O complete")
p.transition("RUNNING", "re-dispatched")
p.transition("TERMINATED", "exit() called")

```
╔═════════════════════════════════════════════════════════╗
║           PCB — Process Control Block                   ║
╠═════════════════════════════════════════════════════════╣
║                                                         ║
║  ┌─────────────────────────────────────────┐            ║
║  │  PID (Process ID)              : 101    │            ║
║  │  State                         : READY  │            ║
║  │  Program Counter (PC)          : 0x4020 │            ║
║  │  CPU Registers                 : {...}  │            ║
║  │  Memory Limits (base, limit)   : (...)  │            ║
║  │  List of Open Files            : [...]  │            ║
║  │  Scheduling Info (priority)    : 5      │            ║
║  │  I/O Status                    : idle   │            ║
║  │  Parent PID (PPID)             : 1      │            ║
║  └─────────────────────────────────────────┘            ║
║                                                         ║
║  PCB is saved/restored during context switch!           ║
║                                                         ║
╚═════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 2. Context Switch Simulation
# ============================================

class PCB:
    def __init__(self, pid, pc, registers):
        self.pid = pid
        self.pc = pc
        self.registers = registers.copy()

class CPU:
    def __init__(self):
        self.current_pid = None
        self.pc = 0
        self.registers = {"AX": 0, "BX": 0}
    
    def context_switch(self, old_pcb: PCB, new_pcb: PCB):
        print(f"\n--- Context Switch: PID {old_pcb.pid} → PID {new_pcb.pid} ---")
        
        # Step 1: Save current state to old PCB
        old_pcb.pc = self.pc
        old_pcb.registers = self.registers.copy()
        print(f"  💾 Saved PID {old_pcb.pid}: PC={old_pcb.pc}, Regs={old_pcb.registers}")
        
        # Step 2: Load new process state
        self.pc = new_pcb.pc
        self.registers = new_pcb.registers.copy()
        self.current_pid = new_pcb.pid
        print(f"  📥 Loaded PID {new_pcb.pid}: PC={self.pc}, Regs={self.registers}")


cpu = CPU()

# Process A was running, modified registers
pcb_a = PCB(pid=101, pc=0x4020, registers={"AX": 42, "BX": 100})
pcb_b = PCB(pid=102, pc=0x8000, registers={"AX": 7, "BX": 99})

cpu.pc = pcb_a.pc
cpu.registers = pcb_a.registers.copy()
cpu.current_pid = 101

# Simulate work on PID 101
cpu.registers["AX"] = 55  # modified during execution
cpu.pc = 0x4030
print(f"CPU running PID 101: PC={cpu.pc}, Regs={cpu.registers}")

# TIME QUANTUM EXPIRED → context switch!
cpu.context_switch(pcb_a, pcb_b)
print(f"\nCPU now running: PID {cpu.current_pid}")

In [ ]:
# ============================================
# 3. fork() — Process Creation
# ============================================
import multiprocessing
import os

def child_task(name):
    print(f"  Child '{name}': PID={os.getpid()}, Parent PID={os.getppid()}")

print(f"Main process: PID={os.getpid()}")

# multiprocessing (cross-platform equivalent of fork)
p1 = multiprocessing.Process(target=child_task, args=("Worker-1",))
p2 = multiprocessing.Process(target=child_task, args=("Worker-2",))

p1.start()
p2.start()
p1.join()  # wait for child to finish
p2.join()

print(f"\nBoth children done. Main still PID={os.getpid()}")

In [ ]:
# ============================================
# 4. Classic fork() Puzzle (Conceptual)
# ============================================

# Q: How many processes are created by 3 consecutive fork() calls?
#    fork()  → 2 processes
#    fork()  → 4 processes  (each of 2 forks again)
#    fork()  → 8 processes  (each of 4 forks again)
# Answer: 2^n = 2^3 = 8 total processes

# Q: How many times is "hello" printed?
# fork()
# print("hello")
# fork()
# print("hello")
#
# After first fork: 2 processes
# Each prints "hello" → 2 prints
# After second fork: 4 processes
# Each prints "hello" → 4 prints
# Total: 6 prints of "hello"

def fork_puzzle():
    # Simulating with a counter
    processes = 1
    total_prints = 0
    
    # fork()
    processes *= 2
    
    # print("hello") — all current processes print
    total_prints += processes
    
    # fork()
    processes *= 2
    
    # print("hello") — all current processes print
    total_prints += processes
    
    print(f"Total processes: {processes}")
    print(f"Total prints:    {total_prints}")

fork_puzzle()

```
╔══════════════════════════════════════════════════════════╗
║          ZOMBIE vs ORPHAN PROCESSES                      ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  ZOMBIE:                                                 ║
║  • Child finished, but parent hasn't called wait()       ║
║  • Entry still in process table (wasting a PID slot)     ║
║  • Fix: parent calls wait()/waitpid()                    ║
║                                                          ║
║  ORPHAN:                                                 ║
║  • Parent terminated before child                        ║
║  • Child adopted by init (PID 1) or systemd              ║
║  • init automatically reaps orphans                      ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Program vs Process vs Thread? | Program=code on disk. Process=running instance+state. Thread=lightweight unit of execution within process |
| 2 | What's in a PCB? | PID, state, PC, registers, memory limits, open files, scheduling info, parent PID |
| 3 | What happens during context switch? | Save current PCB → Load new PCB → Switch memory maps → Resume new process. Pure overhead! |
| 4 | Zombie vs Orphan? | Zombie: child done, parent didn't wait(). Orphan: parent died, child adopted by init |
| 5 | fork() returns what? | Returns child PID to parent, returns 0 to child. Returns -1 on failure |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Process = program in execution with its own state     │
## │  • 5 states: NEW → READY → RUNNING → WAITING → TERM    │
## │  • PCB stores everything about a process                 │
## │  • Context switch = save old PCB + load new PCB          │
## │  • fork() creates child (copy of parent)                 │
## │  • Zombie = done child, lazy parent. Orphan = dead parent│
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Threads** — lightweight processes and the GIL